## 多输入多输出通道
当我们添加通道时，我们的输入和隐藏的表示都变成了三维张量。例如，每个RGB输入图像具有$3*h*w$的形状。我们将这个大小为3的轴称为通道（channel）维度。本节将更深入地研究具有多输入和多输出通道的卷积核。
### 多输入通道
当输入包含多个通道时，需要构造一个与输入数据具有相同输入通道数的卷积核，以便与输入数据进行互相关运算。为了加深理解，我们实现一下多输入通道互相关运算。 简而言之，我们所做的就是对每个通道执行互相关操作，然后将结果相加。

In [7]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X,K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K))
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X,K)

tensor([[ 56.,  72.],
        [104., 120.]])

### 多输出通道
在最流行的神经网络架构中，随着神经网络层数的加深，我们常会增加输出通道的维数，通过减少空间分辨率以获得更大的通道深度。直观地说，我们可以将每个通道看作对不同特征的响应。而现实可能更为复杂一些，因为每个通道不是独立学习的，而是为了共同使用而优化的。因此，多输出通道并不仅是学习多个单通道的检测器。如下所示，我们实现一个计算多个通道的输出的互相关函数。

In [8]:
def corr2d_multi_in_out(X,K):
    # 迭代K的第0个维度，每次都对输入X执行互相关运算，最后将所有结果都叠在一起
    return torch.stack([corr2d_multi_in(X,k) for k in K],0)
# 通过将张量K与K+1和K+2连接起来，构造了一个具有3个输出通道的卷积核
K = torch.stack((K,K+1,K+2),0)
K.shape
# 我们对输入张量X与卷积核张量K执行互相关运算。现在的输出包含
# 个通道，第一个通道的结果与先前输入张量X和多输入单输出通道的结果一致。
corr2d_multi_in_out(X,K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

### 1x1卷积层
因为使用了最小窗口，1x1卷积失去了卷积层的特有能力——在高度和宽度维度上，识别相邻元素间相互作用的能力。 其实
1x1卷积的唯一计算发生在通道上。下面，我们使用全连接层实现1x1卷积。 请注意，我们需要对输入和输出的数据形状进行调整。

In [25]:
def corr2d_multi_in_out_1x1(X,K):
    c_i,h,w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i,h*w)) # 把高和宽拉成二维的
    K = K.reshape((c_o,c_i))
    # 全连接层的矩阵乘法
    Y = torch.matmul(K,X) #列数匹配才能相乘
    return Y.reshape((c_o,h,w))
X = torch.normal(0,1,(3,3,3))
K = torch.normal(0, 1, (2, 3, 1, 1))
    
Y1 = corr2d_multi_in_out_1x1(X,K)
Y2 = corr2d_multi_in_out(X,K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6


### 小结
多输入多输出通道可以用来扩展卷积层的模型。

当以每像素为基础应用时，1x1卷积层相当于全连接层。

1x1卷积层通常用于调整网络层的通道数量和控制模型复杂性。